# Save outputs to drive
Don't run this cell initially. Instead, after you've completed an experiment and obtained a result you can run it to save that result to a unique folder in drive (so that it doesn't overwrite old saved data)

In [ ]:
drive.mount('/content/drive')

current_time = datetime.now()
print(current_time)

outputs_foldername = f"BlorbMyGlorb {current_time}"

shutil.copytree("outputs", f"/content/drive/MyDrive/{outputs_foldername}")

with open(f"/content/drive/MyDrive/{outputs_foldername}/output_info.txt","w") as info_file:
  info_file.write(f"Prompt: {PROMPT_TEMPLATE_TF}\n")
  info_file.write(f"Datasets: {', '.join(DATASET_NAMES)}\n")

In [ ]:
!pip install -q torch transformers accelerate pandas pyarrow scikit-learn matplotlib huggingface_hub bitsandbytes

In [ ]:
from __future__ import annotations
import json
from pathlib import Path

# Numpy and Pandas double wammy
import numpy as np
import pandas as pd

# Machine learning util
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# Used to gain access to linux shell commands (e.g. !cp)
import shutil

# Misc util
from google.colab import drive
from datetime import datetime

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")

In [ ]:
from google.colab import files

def upload_and_convert():
    """Upload a single dataset file and convert to parquet. Returns (parquet_path, base_name)."""
    print("Upload your dataset (.csv, .parquet, or .jsonl):")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    print(f"Uploaded: {filename}")

    if filename.endswith(".parquet"):
        df_raw = pd.read_parquet(filename)
    elif filename.endswith(".jsonl"):
        df_raw = pd.read_json(filename, lines=True)
    elif filename.endswith(".csv"):
        df_raw = pd.read_csv(filename)
    else:
        raise ValueError(f"Unsupported file type: {filename}")

    print(f"Raw columns: {list(df_raw.columns)}")
    print(f"Raw shape: {df_raw.shape}")
    print(df_raw.head())

    raw_labels = df_raw["label"]
    if np.issubdtype(raw_labels.dtype, np.integer):
        pass
    elif raw_labels.dtype == bool:
        df_raw["label"] = raw_labels.astype(int)
    else:
        label_map = {"true": 1, "false": 0, "True": 1, "False": 0, True: 1, False: 0}
        df_raw["label"] = raw_labels.map(label_map)
        unmapped = df_raw["label"].isna().sum()
        if unmapped > 0:
            print(f"WARNING: {unmapped} rows had unrecognized labels, dropping them")
            df_raw = df_raw.dropna(subset=["label"])
        df_raw["label"] = df_raw["label"].astype(int)

    base_name = filename.rsplit(".", 1)[0]

    if "id" not in df_raw.columns:
        df_raw["id"] = [f"{base_name}_{i:06d}" for i in range(len(df_raw))]
    if "domain" not in df_raw.columns:
        df_raw["domain"] = base_name

    parquet_path = base_name + ".parquet"
    df_raw.to_parquet(parquet_path, index=False)
    print(f"\nSaved to {parquet_path}")
    print(f"  {len(df_raw)} rows, label distribution: {df_raw['label'].value_counts().to_dict()}")
    return parquet_path, base_name

DATASET_PATHS = []
DATASET_NAMES = []

for i in range(3):
    print(f"\n{'='*50}")
    print(f"--- Upload dataset {i+1}/3 ---")
    print(f"{'='*50}")
    path, name = upload_and_convert()
    DATASET_PATHS.append(path)
    DATASET_NAMES.append(name)

print(f"\nAll datasets ready: {DATASET_NAMES}")

In [ ]:
# ---- CONFIG ----
MODEL_NAME = "meta-llama/Llama-2-13b-hf"   # change to any HF model
FEATURES_DIR = "features"
BATCH_SIZE = 4
MAX_LENGTH = 256
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PROMPT_TEMPLATE = "Answer True/False: {statement}"
PROBE_C = 1.0
TEST_SIZE = 0.2
SEED = 42
N_PAIRS = 10000
COV_REG = 1e-4
USE_4BIT = False  # 4-bit quantization — required for 13B models on T4 (16 GB VRAM)

print(f"Model: {MODEL_NAME}")
print(f"Device: {DEVICE}")
print(f"4-bit quantization: {USE_4BIT}")
print(f"Batch size: {BATCH_SIZE}")

In [ ]:
from huggingface_hub import login
login()  # paste your HuggingFace token when prompted

# Utility Functions

In [ ]:
def _infer_input_device(model, fallback_device: str) -> torch.device:
    """ Just sets up CUDA shenanigans """
    if hasattr(model, "hf_device_map") and isinstance(model.hf_device_map, dict):
        for key in ["model.embed_tokens", "embed_tokens", "lm_head"]:
            if key in model.hf_device_map:
                dev = model.hf_device_map[key]
                if isinstance(dev, int):
                    return torch.device(f"cuda:{dev}")
                return torch.device(str(dev))
        first_dev = next(iter(model.hf_device_map.values()), None)
        if first_dev is not None:
            if isinstance(first_dev, int):
                return torch.device(f"cuda:{first_dev}")
            return torch.device(str(first_dev))
    return torch.device(fallback_device)

def load_model(model_name, device, use_4bit=False):
    """Load the model and tokenizer once. Returns (model, tokenizer, input_device)."""
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    dtype = torch.float16 if device.startswith("cuda") else torch.float32

    if use_4bit and device.startswith("cuda"):
        print("Loading model with 4-bit quantization (NF4)...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name, quantization_config=bnb_config, device_map="auto",
        )
    elif device.startswith("cuda"):
        print(f"Loading model in {dtype}...")
        model = AutoModelForCausalLM.from_pretrained(
            model_name, dtype=dtype, device_map="auto",
        )
    else:
        print(f"Loading model in {dtype}...")
        model = AutoModelForCausalLM.from_pretrained(model_name, dtype=dtype)
        model.to(device)
    model.eval()
    input_device = _infer_input_device(model, fallback_device=device)
    print(f"Model loaded. Input device: {input_device}")
    return model, tokenizer, input_device


def model_output(model, tokenizer, input_device, dataset_path, out_dir,
                 batch_size, max_length, prompt_template):
    """Generate the model's next-token for each prompt and classify as true/false.

    Saved to CSV: prompt_num, answer, output, hallucination, confidence
    """

    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    df = pd.read_parquet(dataset_path)
    required = {"id", "domain", "label", "statement"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {', '.join(sorted(missing))}")

    texts = [prompt_template.format(statement=str(s)) for s in df["statement"].tolist()]
    ground_truth = df["label"].astype(int).to_numpy()
    print(f"Loaded {len(texts)} prompts")

    true_token_ids = [tokenizer.encode(v, add_special_tokens=False)[0]
                      for v in ["True", "true", " True", " true"]]
    false_token_ids = [tokenizer.encode(v, add_special_tokens=False)[0]
                       for v in ["False", "false", " False", " false"]]

    results = []
    total_batches = (len(texts) + batch_size - 1) // batch_size

    all_batch_activations = []

    for batch_num, i in enumerate(range(0, len(texts), batch_size)):
        if batch_num % 20 == 0:
            print(f"  Batch {batch_num + 1}/{total_batches}")
        batch_texts = texts[i : i + batch_size]

        enc = tokenizer(
            batch_texts, return_tensors="pt", padding=True,
            truncation=True, max_length=max_length,
        )
        enc = {k: v.to(input_device) for k, v in enc.items()}

        with torch.no_grad():
            generated_ids = model.generate(
                **enc,
                max_new_tokens=1,
                do_sample=False,
                output_hidden_states=True,
                output_scores=True,
                return_dict_in_generate=True,
            )

        new_token_ids = generated_ids.sequences[:, enc["input_ids"].shape[1]:]
        decoded_tokens = tokenizer.batch_decode(new_token_ids, skip_special_tokens=True)
        hs = generated_ids.hidden_states

        scores = generated_ids.scores[0]
        probs = torch.softmax(scores.float(), dim=-1)
        p_true = probs[:, true_token_ids].sum(dim=-1)
        p_false = probs[:, false_token_ids].sum(dim=-1)
        p_total = p_true + p_false
        confidence = torch.where(p_true >= p_false, p_true, p_false) / p_total.clamp(min=1e-8)
        confidence = confidence.cpu().numpy()

        target_layers = torch.stack([h[:, -1, :] for h in hs[0][12:17]]).cpu().numpy()
        all_batch_activations.append(target_layers)

        for j, raw_output in enumerate(decoded_tokens):
            idx = i + j
            output_stripped = raw_output.strip()
            output_lower = output_stripped.lower()

            if output_lower == "true":
                model_answer = 1
            elif output_lower == "false":
                model_answer = 0
            else:
                model_answer = -1

            gt = int(ground_truth[idx])
            gt_str = "True" if gt == 1 else "False"

            if model_answer == -1:
                hallucinated = "N/A"
            else:
                hallucinated = "Yes" if model_answer != gt else "No"

            results.append({
                "prompt_num": idx,
                "answer": gt_str,
                "output": output_stripped,
                "hallucination": hallucinated,
                "confidence": round(float(confidence[j]), 4),
            })

    # Save the target intermediate layer activations to an npz file for later analysis
    all_batch_activations = np.concatenate(all_batch_activations, axis=1)
    np.savez_compressed(
        out_path / "hs_12_to_17.npz",
        activations=all_batch_activations
    )

    # Big D's Code code
    results_df = pd.DataFrame(results)
    csv_path = out_path / "model_outputs.csv"
    results_df.to_csv(csv_path, index=False)
    print(f"\nSaved {len(results_df)} results to {csv_path}")

    valid = results_df[results_df["hallucination"] != "N/A"]
    if len(valid) > 0:
        n_hall = (valid["hallucination"] == "Yes").sum()
        print(f"Hallucination rate: {n_hall}/{len(valid)} ({100 * n_hall / len(valid):.1f}%)")
    n_na = (results_df["hallucination"] == "N/A").sum()
    if n_na > 0:
        print(f"Unrecognized outputs (not true/false): {n_na}")
        print("Sample unrecognized:", results_df[results_df["hallucination"] == "N/A"]["output"].value_counts().head(10).to_dict())

    return results_df



PROMPT_TEMPLATE_TF = "True or False: {statement}\nAnswer:"

# Load model ONCE
model, tokenizer, input_device = load_model(MODEL_NAME, DEVICE, USE_4BIT)

# Run inference on all 3 datasets
all_results = {}
for ds_path, ds_name in zip(DATASET_PATHS, DATASET_NAMES):
    print(f"\n{'='*60}")
    print(f"Processing: {ds_name}")
    print(f"{'='*60}")
    out_dir = f"outputs/{ds_name}"
    output_df = model_output(
        model=model,
        tokenizer=tokenizer,
        input_device=input_device,
        dataset_path=ds_path,
        out_dir=out_dir,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        prompt_template=PROMPT_TEMPLATE_TF,
    )
    all_results[ds_name] = output_df

    print("\n--- First 10 rows ---")
    print(output_df.head(10).to_string(index=False))
    print(f"\n--- Output value counts ---")
    print(output_df["output"].value_counts().head(15))
    print(f"\n--- Hallucination breakdown ---")
    print(output_df["hallucination"].value_counts())

print(f"\nAll 3 datasets processed.")


In [ ]:
from sklearn.decomposition import PCA
from enum import Enum
import matplotlib.pyplot as plt
import matplotlib.cm as cm

class answer_status(Enum):
  TRUE_POSITIVE = 0
  TRUE_NEGATIVE = 1
  FALSE_POSITIVE = 2
  FALSE_NEGATIVE = 3
  NOT_AVAILABLE = 4

layer = 3  # acts[3] = layer 15

fig, axes = plt.subplots(3, 4, figsize=(24, 18))
col_titles = ["Only Hallucinations", "All", "Non-Hallucinations", "Hallucination Model Confidence"]

for row_idx, ds_name in enumerate(DATASET_NAMES):
    out_dir = f"outputs/{ds_name}"
    mo = pd.read_csv(f"{out_dir}/model_outputs.csv")
    data = np.load(f"{out_dir}/hs_12_to_17.npz")
    acts = data["activations"]

    X = acts[layer]
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)

    # Build answer_status array (same logic as original cell 10)
    answer_statuses = []
    for _, row in mo.iterrows():
        h = row["hallucination"]
        a = row["output"]
        if h == "No" and a == "True":
            answer_statuses.append(answer_status.TRUE_POSITIVE)
        elif h == "Yes" and a == "True":
            answer_statuses.append(answer_status.FALSE_POSITIVE)
        elif h == "No" and a == "False":
            answer_statuses.append(answer_status.TRUE_NEGATIVE)
        elif h == "Yes" and a == "False":
            answer_statuses.append(answer_status.FALSE_NEGATIVE)
        else:
            answer_statuses.append(answer_status.NOT_AVAILABLE)

    as_vals = np.array([s.value for s in answer_statuses])
    true_mask = (as_vals == 0) | (as_vals == 1)
    false_mask = (as_vals == 2) | (as_vals == 3)
    hal_mask = (mo["hallucination"] == "Yes").to_numpy()
    non_hal_mask = (mo["hallucination"] == "No").to_numpy()
    conf = mo["confidence"].to_numpy()

    # Column 0: Only Hallucinations (blue = said True wrong, red = said False wrong)
    ax0 = axes[row_idx, 0]
    if false_mask.sum() > 0:
        hal_pca = X_pca[false_mask]
        hal_as = as_vals[false_mask]
        hal_colors = ["blue" if s == answer_status.FALSE_POSITIVE.value else "red" for s in hal_as]
        ax0.scatter(hal_pca[:, 0], hal_pca[:, 1], c=hal_colors, s=10, alpha=0.7)
    ax0.set_xlabel("PC1")
    ax0.set_ylabel("PC2")

    # Column 1: All (correct=Blues cmap, hallucinated=Reds cmap 'x')
    ax1 = axes[row_idx, 1]
    if non_hal_mask.sum() > 0:
        ax1.scatter(X_pca[non_hal_mask, 0], X_pca[non_hal_mask, 1],
                    c=conf[non_hal_mask], cmap="Blues", s=10, alpha=0.5, label="Correct")
    if hal_mask.sum() > 0:
        ax1.scatter(X_pca[hal_mask, 0], X_pca[hal_mask, 1],
                    c=conf[hal_mask], cmap="Reds", s=20, marker="x", alpha=0.7, label="Hallucinated")
    ax1.legend(fontsize=7)
    ax1.set_xlabel("PC1")
    ax1.set_ylabel("PC2")

    # Column 2: Non-Hallucinations (blue = said True correct, red = said False correct)
    ax2 = axes[row_idx, 2]
    if true_mask.sum() > 0:
        truth_pca = X_pca[true_mask]
        truth_as = as_vals[true_mask]
        truth_colors = ["blue" if s == answer_status.TRUE_POSITIVE.value else "red" for s in truth_as]
        ax2.scatter(truth_pca[:, 0], truth_pca[:, 1], c=truth_colors, s=10, alpha=0.7)
    ax2.set_xlabel("PC1")
    ax2.set_ylabel("PC2")

    # Column 3: Hallucination Model Confidence (coolwarm)
    ax3 = axes[row_idx, 3]
    if hal_mask.sum() > 0:
        sc = ax3.scatter(X_pca[hal_mask, 0], X_pca[hal_mask, 1],
                         c=conf[hal_mask], cmap="coolwarm", s=25, alpha=0.8)
        plt.colorbar(sc, ax=ax3, label="Confidence")
    ax3.set_xlabel("PC1")
    ax3.set_ylabel("PC2")

    # Row label on leftmost column
    axes[row_idx, 0].set_ylabel(f"{ds_name}\nPC2", fontsize=12, fontweight="bold")

# Column titles on top row
for col_idx, title in enumerate(col_titles):
    axes[0, col_idx].set_title(title, fontsize=13, fontweight="bold")

plt.tight_layout()
plt.savefig("outputs/combined_pca_grid.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: outputs/combined_pca_grid.png")


In [ ]:
def conf_stats(values, label):
    if len(values) == 0:
        print(f"  {label}: no data")
        return
    q1, median, q3 = np.percentile(values, [25, 50, 75])
    iqr = q3 - q1
    print(f"  {label}:")
    print(f"    N={len(values)}, Mean={values.mean():.4f}, Std={values.std():.4f}")
    print(f"    Median={median:.4f}, Q1={q1:.4f}, Q3={q3:.4f}, IQR={iqr:.4f}")
    print(f"    Min={values.min():.4f}, Max={values.max():.4f}")

for ds_name in DATASET_NAMES:
    out_dir = f"outputs/{ds_name}"
    mo = pd.read_csv(f"{out_dir}/model_outputs.csv")

    hal_mask = (mo["hallucination"] == "Yes").to_numpy()
    non_hal_mask = (mo["hallucination"] == "No").to_numpy()
    conf = mo["confidence"].to_numpy()

    print(f"\n{'='*60}")
    print(f"Dataset: {ds_name}")
    print(f"{'='*60}")
    print(f"  Total: {len(mo)}, Hallucinated: {hal_mask.sum()}, "
          f"Non-hallucinated: {non_hal_mask.sum()}, "
          f"N/A: {(mo['hallucination'] == 'N/A').sum()}")
    print()
    conf_stats(conf, "All")
    conf_stats(conf[hal_mask], "Hallucinated")
    conf_stats(conf[non_hal_mask], "Non-hallucinated")

    if hal_mask.sum() > 0 and non_hal_mask.sum() > 0:
        print(f"\n  Median diff (non-hal - hal): "
              f"{np.median(conf[non_hal_mask]) - np.median(conf[hal_mask]):.4f}")
        print(f"  Mean diff (non-hal - hal): "
              f"{conf[non_hal_mask].mean() - conf[hal_mask].mean():.4f}")


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.decomposition import PCA

layer = 3  # acts[3] = layer 15

all_probe_rows = []

for ds_name in DATASET_NAMES:
    out_dir = f"outputs/{ds_name}"
    mo = pd.read_csv(f"{out_dir}/model_outputs.csv")
    data = np.load(f"{out_dir}/hs_12_to_17.npz")
    acts = data["activations"]

    # Filter to hallucination-only examples
    hal_mask = (mo["hallucination"] == "Yes").to_numpy()
    hal_outputs = mo[hal_mask]["output"].to_numpy()

    # y = 1 if hallucinated "True", y = 0 if hallucinated "False"
    hal_said_true = (hal_outputs == "True")
    hal_said_false = (hal_outputs == "False")
    recognized = hal_said_true | hal_said_false

    X_hal = acts[layer][hal_mask][recognized]
    y_hal = hal_said_true[recognized].astype(int)

    n_true = y_hal.sum()
    n_false = len(y_hal) - n_true

    print(f"\n{'='*60}")
    print(f"Dataset: {ds_name}")
    print(f"{'='*60}")
    print(f"  Total hallucinations: {hal_mask.sum()}")
    print(f"  Recognized outputs: {recognized.sum()}")
    print(f"  Said True (wrong): {n_true}")
    print(f"  Said False (wrong): {n_false}")

    if n_true < 2 or n_false < 2:
        print(f"  SKIPPING — not enough examples in both classes for train/test split")
        all_probe_rows.append({
            "dataset": ds_name, "layer": layer + 12,
            "n_hal": int(hal_mask.sum()), "n_said_true": int(n_true), "n_said_false": int(n_false),
            "train_bal_acc": None, "train_auroc": None,
            "test_bal_acc": None, "test_auroc": None,
        })
        continue

    X_train, X_test, y_train, y_test = train_test_split(
        X_hal, y_hal, test_size=0.2, random_state=42, stratify=y_hal
    )

    clf = LogisticRegression(max_iter=200, solver="lbfgs", class_weight="balanced", C=1.0)
    clf.fit(X_train, y_train)

    train_probs = clf.predict_proba(X_train)[:, 1]
    train_preds = (train_probs >= 0.5).astype(int)
    train_acc = balanced_accuracy_score(y_train, train_preds)
    train_auroc = roc_auc_score(y_train, train_probs)

    test_probs = clf.predict_proba(X_test)[:, 1]
    test_preds = (test_probs >= 0.5).astype(int)
    test_acc = balanced_accuracy_score(y_test, test_preds)
    test_auroc = roc_auc_score(y_test, test_probs)

    print(f"  Train balanced acc: {train_acc:.4f}  Train AUROC: {train_auroc:.4f}")
    print(f"  Test  balanced acc: {test_acc:.4f}  Test  AUROC: {test_auroc:.4f}")

    all_probe_rows.append({
        "dataset": ds_name, "layer": layer + 12,
        "n_hal": int(hal_mask.sum()), "n_said_true": int(n_true), "n_said_false": int(n_false),
        "train_bal_acc": round(train_acc, 4), "train_auroc": round(train_auroc, 4),
        "test_bal_acc": round(test_acc, 4), "test_auroc": round(test_auroc, 4),
    })

hal_probe_df = pd.DataFrame(all_probe_rows)
hal_probe_df.to_csv("outputs/hallucination_true_vs_false_probe.csv", index=False)
print(f"\nSaved to outputs/hallucination_true_vs_false_probe.csv")
print()
hal_probe_df

In [ ]:
# (removed — superseded by combined grid above)

In [ ]:
# (removed — superseded by combined grid above)

In [ ]:
# (removed — superseded by combined grid above)

In [ ]:
# (removed — superseded by combined grid above)

In [ ]:
# (removed — superseded by combined grid above)